# 01 — Sparse Autoencoders (Cunningham et al.)

Train a **tied SAE**, plot **FVU vs L0** across L1 (α), and inspect top features.

Docs: `docs/sae_reimpl/` · Paper: [arXiv:2309.08600](https://arxiv.org/abs/2309.08600)

## 0. Colab / local setup

In [ ]:
# --- Colab / local bootstrap (Drive + wheels + swappable model) ---
# Drive layout expected:
#   MyDrive/multilingual-mechinterp/
#     dist/*.whl
#     data/all200questions_persianMiddleEastCulture.json
#     configs/  notebooks/  results/
#
# Edit MODEL_NAME in notebooks/colab_setup.py (Qwen2.5 now; Gemma later),
# or override below after bootstrap.

from pathlib import Path
import runpy

def _resolve_setup_script() -> Path:
    here = Path.cwd()
    candidates = [
        here / "colab_setup.py",
        here / "notebooks" / "colab_setup.py",
        here.parent / "notebooks" / "colab_setup.py",
        Path("/content/drive/MyDrive/multilingual-mechinterp/notebooks/colab_setup.py"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "colab_setup.py not found. Mount Drive with the project folder, "
        "or open the notebook from the repo."
    )

_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

# Session overrides (uncomment as needed):
# MODEL_NAME = "google/gemma-2-2b"
# MODEL_TRUST_REMOTE_CODE = False
# USE_TINY_OFFLINE = True   # demos without downloading HF weights

import matplotlib.pyplot as plt
import torch

from multilingual_mechinterp.utils import ensure_dir, load_config

cfg_path = CONFIG_DIR / "qwen25.yaml"
cfg = load_config(cfg_path) if cfg_path.exists() else {}
if "model" in cfg and not USE_TINY_OFFLINE:
    # keep notebook MODEL_NAME as source of truth; cfg is fallback metadata
    pass

print("Ready.")
print(" ROOT =", ROOT)
print(" DATA =", DATA_DIR)
print(" DIST =", DIST_DIR)
print(" MODEL =", MODEL_NAME, "| tiny=", USE_TINY_OFFLINE)

from multilingual_mechinterp.metrics import evaluate_sae
from multilingual_mechinterp.sae import TiedSAE, sweep_l1, train_tied_sae

OUT = ensure_dir(RESULTS_DIR / "sae")
sae_cfg = cfg.get("sae", {"ratio": 2, "alpha": 8.6e-4, "lr": 1e-3,
                          "alphas": [1e-4, 3e-4, 8.6e-4, 1.4e-3, 3e-3]})
sae_cfg


## Load experiment model

Uses `MODEL_NAME` from `colab_setup.py` (default **Qwen2.5**). Set `USE_TINY_OFFLINE=True` for demos without HF downloads.


In [ ]:
# Real model (Qwen now; change MODEL_NAME for Gemma later) OR tiny offline
# model = load_experiment_model()
# For gated Gemma: export HF_TOKEN=... or pass token=...

# Default path in analysis cells below uses tiny models for speed.
# Swap in `model = load_experiment_model()` when you are ready for Qwen/Gemma.
print("To load HF weights:", f"load_experiment_model({MODEL_NAME!r})")
print("Culture JSON:", culture_json_path(), "exists=", culture_json_path().exists())


## 1. Offline demo (no HF download)

Synthetic residual activations with Pythia-like width `d=512`.

In [ ]:
torch.manual_seed(0)
d = 64  # use 512 for a heavier demo
acts = torch.randn(4000, d)

single = train_tied_sae(
    acts,
    ratio=cfg["sae"]["ratio"],
    alpha=cfg["sae"]["alpha"],
    lr=cfg["sae"]["lr"],
    batch_size=256,
    n_epochs=2,
    device="cpu",
)
metrics = evaluate_sae(single.sae, acts[:2000])
print(metrics)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([h["loss"] for h in single.history], marker="o")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Tied SAE training loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. L1 (α) sweep — FVU vs sparsity (paper Fig. 2 style)

In [ ]:
alphas = cfg["sae"].get("alphas", [1e-4, 3e-4, 8.6e-4, 1.4e-3, 3e-3])
rows = sweep_l1(
    acts,
    alphas=alphas,
    ratio=cfg["sae"]["ratio"],
    output_dir=OUT / "l1_sweep_demo",
    lr=cfg["sae"]["lr"],
    batch_size=256,
    n_epochs=1,
    device="cpu",
)

fvu = [r["fvu"] for r in rows]
l0 = [r["mean_l0"] for r in rows]
dead = [r["dead_features"] for r in rows]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(fvu, l0, marker="o")
for r, x, y in zip(rows, fvu, l0):
    axes[0].annotate(f"α={r['alpha']:.1e}", (x, y), fontsize=8, xytext=(4, 4),
                     textcoords="offset points")
axes[0].set_xlabel("FVU (lower = better recon)")
axes[0].set_ylabel("mean L0 (active features)")
axes[0].set_title("FVU–sparsity Pareto")
axes[0].grid(True, alpha=0.3)

axes[1].bar([f"{r['alpha']:.0e}" for r in rows], dead, color="#4C78A8")
axes[1].set_xlabel("α")
axes[1].set_ylabel("dead features")
axes[1].set_title("Dead features vs α")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(OUT / "fvu_l0_pareto.png", dpi=150)
plt.show()
rows

## 3. Inspect top activating features on a sample

In [ ]:
sae = TiedSAE.load(rows[len(rows)//2]["checkpoint"])
x = acts[:512]
with torch.no_grad():
    c = sae.encode(x)
mean_act = c.mean(0)
topk = torch.topk(mean_act, k=15)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([str(int(i)) for i in topk.indices], topk.values.numpy(), color="#F58518")
ax.set_xlabel("feature id")
ax.set_ylabel("mean activation")
ax.set_title("Top-15 features by mean activation")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(OUT / "top_features.png", dpi=150)
plt.show()

## 4. Real Pythia run (optional)

```bash
python scripts/extract_activations.py --config configs/pythia70m_sae.yaml --device cuda
python scripts/train_sae_l1_sweep.py --config configs/pythia70m_sae.yaml --device cuda
```
Then reload `results/sae/l1_sweep/sweep_metrics.json` and replot.